In [5]:
import time
import io
from PIL import Image
from transformers import pipeline
from google.colab import files

# Loading models
print("Loading models into memory\n")
print("-" * 50)

# Model 1 : The Specialist Pokemon Classifier
specialist_pipe = pipeline("image-classification", model="imjeffhi/pokemon_classifier")

# Model 2 : The Generalist Image Classifier
zero_shot_pipe = pipeline("zero-shot-image-classification", model="openai/clip-vit-base-patch32")

# For Zero-Shot, we are giving it a list of possibilities
possible_pokemon = [
    "Pikachu", "Charizard", "Bulbasaur", "Squirtle", "Eevee",
    "Mewtwo", "Gengar", "Snorlax", "Lucario", "Jigglypuff",
    "Dragonite", "Rayquaza", "Greninja", "Togepi", "Machamp",
    "Gyarados", "Arcanine", "Umbreon", "Lugia", "Ditto"
]

# Model 3 : The BLIP Vision-Language Model
vqa_pipe = pipeline("visual-question-answering", model="Salesforce/blip-vqa-base")

# Loading the images
print("Models have been loaded. Upload the images")
uploaded = files.upload()

print("-" * 50)

# Looping through each uploaded image one by one
for image_filename in uploaded.keys():
    print(f"\n Testing Image: {image_filename} ---")

    try:
        image = Image.open(io.BytesIO(uploaded[image_filename])).convert("RGB")
    except Exception as e:
        print(f"Error loading image: {e}")
        continue # Skips to the next image if there is an error

    # Testing Model 1: Specialist
    start_time = time.time()
    specialist_result = specialist_pipe(image)
    specialist_time = time.time() - start_time

    # It returns a list of dictionaries and we want the top prediction
    top_specialist_pred = specialist_result[0]['label']
    top_specialist_score = specialist_result[0]['score']
    print(f"Model 1 (Specialist): {top_specialist_pred} (Confidence: {top_specialist_score:.1%}) - Time: {specialist_time:.2f}s")

    # Testing Model 2: Generalist
    start_time = time.time()
    zero_shot_result = zero_shot_pipe(image, candidate_labels=possible_pokemon)
    zero_shot_time = time.time() - start_time

    top_zs_pred = zero_shot_result[0]['label']
    top_zs_score = zero_shot_result[0]['score']
    print(f"Model 2 (Generalist): {top_zs_pred} (Confidence: {top_zs_score:.1%}) - Time: {zero_shot_time:.2f}s")

    # Testing Model 3: BLIP VQA
    question = "What is the exact name of this Pokemon character?"
    start_time = time.time()
    vqa_result = vqa_pipe(image, question, top_k=1)
    vqa_time = time.time() - start_time

    # VQA only returns the text answer
    top_vqa_pred = vqa_result[0]['answer'].title()
    print(f"Model 3 (BLIP VQA)  : {top_vqa_pred} - Time: {vqa_time:.2f}s")

print("\n" + "-" * 50)
print("Testing completed. Compare the accuracy, confidence, and speed of each model")

Loading models into memory

--------------------------------------------------


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/788 [00:00<?, ?it/s]

...

Model 3 (BLIP VQA)  : Pikachu - Time: 2.22s

 Testing Image: snor (1).jpg ---
Model 1 (Specialist): Snorlax (Confidence: 99.7%) - Time: 0.51s
Model 2 (Generalist): Snorlax (Confidence: 91.9%) - Time: 0.46s


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Model 3 (BLIP VQA)  : Blue - Time: 2.05s

 Testing Image: test_pokemon (2).jpg ---
Model 1 (Specialist): Eevee (Confidence: 99.5%) - Time: 0.51s
Model 2 (Generalist): Eevee (Confidence: 97.2%) - Time: 0.43s


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Model 3 (BLIP VQA)  : Pikachu - Time: 2.54s

--------------------------------------------------
Testing completed. Compare the accuracy, confidence, and speed of each model
